# Pruebas de hipótesis de las medidas de los componentes ERP

## Funciones

In [22]:
import numpy as np
import pandas as pd
import pingouin as pg
import scipy.stats as stats
import statsmodels.api as sm
from scipy.stats import chi2_contingency
from statsmodels.formula.api import ols
from statsmodels.sandbox.stats.multicomp import multipletests

In [20]:
def test_ancova_full(data, dv, between):
    """
    Realiza un ANCOVA y análisis post-hoc completos, incluyendo pruebas de supuestos,
    y devuelve un DataFrame con los resultados formateados.

    Parámetros:
    - data: DataFrame de pandas con los datos.
    - dv: Nombre de la variable dependiente (string).
    - between: Nombre de la variable categórica independiente (string).

    Devuelve:
    - Un diccionario con el modelo ANCOVA, los resultados de las pruebas de supuestos,
      las medias ajustadas y las comparaciones por pares.
    """
    # --- 1. Pruebas de Supuestos ---
    print("--- 1. Pruebas de Supuestos del ANCOVA ---")
    
    # Supuesto de Normalidad de los Residuos (Shapiro-Wilk)
    # Ajustamos un modelo lineal para obtener los residuos
    formula_residuos = f"{dv} ~ age + {between}"
    modelo_residuos = ols(formula_residuos, data=data).fit()
    residuos = modelo_residuos.resid
    shapiro_test = stats.shapiro(residuos)
    print(f"Prueba de Shapiro-Wilk para la normalidad de los residuos: W={shapiro_test.statistic:.4f}, p-valor={shapiro_test.pvalue:.4f}")
    if shapiro_test.pvalue > 0.05:
        print("  > Conclusión: Los residuos siguen una distribución normal (p > 0.05).")
    else:
        print("  > Conclusión: Los residuos no siguen una distribución normal (p <= 0.05).")

    # Supuesto de Homogeneidad de Varianzas (Prueba de Levene)
    levene_test = pg.homoscedasticity(data=data, dv=dv, group=between)
    levene_w = float(levene_test.iloc[0]['W'])
    levene_p = float(levene_test.iloc[0]['pval'])
    print(f"\nPrueba de Levene para la homogeneidad de varianzas: W={levene_w:.4f}, p-valor={levene_p:.4f}")
    if levene_p > 0.05:
        print("  > Conclusión: Las varianzas son homogéneas entre los grupos (p > 0.05).")
    else:
        print("  > Conclusión: Las varianzas no son homogéneas entre los grupos (p <= 0.05).")

    # Supuesto de Homogeneidad de las Pendientes de Regresión
    formula_interaccion = f"{dv} ~ age + {between} + age:{between}"
    modelo_interaccion = ols(formula_interaccion, data=data).fit()
    resultado_interaccion = sm.stats.anova_lm(modelo_interaccion, typ=2)
    interaction_term = f'age:{between}'
    f_interaccion = resultado_interaccion.at[interaction_term, 'F']
    if isinstance(f_interaccion, complex):
        f_interaccion = f_interaccion.real
    f_raw = cast(Any, f_interaccion)
    p_raw = cast(Any, resultado_interaccion.at[interaction_term, 'PR(>F)'])

    f_interaccion = float(f_raw.real if isinstance(f_raw, complex) else f_raw)
    p_valor_interaccion = float(p_raw.real if isinstance(p_raw, complex) else p_raw)
    print(f"\nPrueba de homogeneidad de las pendientes de regresión (interacción 'age:{between}'):")
    print(f"  > F={f_interaccion:.4f}, p-valor={p_valor_interaccion:.4f}")
    if p_valor_interaccion > 0.05:
        print("  > Conclusión: Se cumple el supuesto de homogeneidad de las pendientes de regresión (p > 0.05).")
    else:
        print("  > Conclusión: No se cumple el supuesto de homogeneidad de las pendientes de regresión (p <= 0.05).")

    # --- 2. Ejecución del ANCOVA ---
    print("\n--- 2. Resultados del ANCOVA ---")
    ancova_model = pg.ancova(data=data, dv=dv, between=between, covar='age')
    print(ancova_model)

    # --- 3. Análisis Post-Hoc: Comparaciones por Pares con Ajuste de Bonferroni ---
    print("\n--- 3. Análisis Post-Hoc: Comparaciones por Pares (Bonferroni) ---")

    # Ajustamos ANCOVA con statsmodels para obtener valores ajustados por edad
    formula_ancova_sm = f"{dv} ~ age + C({between})"
    modelo_ancova_sm = ols(formula_ancova_sm, data=data).fit()

    # Datos usados efectivamente en el modelo (sin NaN en variables relevantes)
    data_modelo = modelo_ancova_sm.model.data.frame.copy()

    # Variable dependiente ajustada por covariable (residualizada)
    data_modelo['dv_ajustada'] = modelo_ancova_sm.resid + data_modelo[dv].mean()

    # Comparaciones por pares (sin covar, porque no es parámetro válido en pairwise_tests)
    pairwise_comparisons = pg.pairwise_tests(
        data=data_modelo,
        dv='dv_ajustada',
        between=between,
        padjust='bonf'
    )
    print(pairwise_comparisons)

    # --- 4. Medias Ajustadas y Errores Estándar ---
    # Medias ajustadas estimadas al valor medio de edad
    mean_age = data_modelo['age'].mean()
    group_labels = sorted(data_modelo[between].dropna().unique())

    pred_df = pd.DataFrame({
        between: group_labels,
        'age': [mean_age] * len(group_labels)
    })

    pred = modelo_ancova_sm.get_prediction(pred_df).summary_frame(alpha=0.05)

    medias_ajustadas = pd.DataFrame({
        'type': group_labels,
        'Media_Ajustada (M_adj)': pred['mean'].round(4).values,
        'Error_Estandar (EE)': pred['mean_se'].round(4).values,
        'IC_95%_Inferior': pred['mean_ci_lower'].round(4).values,
        'IC_95%_Superior': pred['mean_ci_upper'].round(4).values
    })

    print("\n--- Medias Ajustadas, Error Estándar e Intervalos de Confianza (95%) ---")
    print(medias_ajustadas)

    # --- 5. Devolución de Resultados ---
    return {
        'ancova_model': ancova_model,
        'shapiro_test': shapiro_test,
        'levene_test': levene_test,
        'interaction_test': resultado_interaccion,
        'pairwise_comparisons': pairwise_comparisons,
        'adjusted_means': medias_ajustadas
    }


In [3]:
# Función para verificar los supuestos del ANCOVA mediante un modelo con interacciones

def verify_ancova_assumptions(data, dv, between):
    # Cargar el dataframe y eliminar la fila con el valor faltante en IAT_score
    df = data.dropna(subset=[dv, between])

    # Definición y ajuste del modelo de interacción
    ancova_interaction_model = ols( f'{dv} ~ C({between}) * age + C({between}) * scholarship + C({between}) * C(gender)', 
    data=df).fit()

    # --- VERIFICACIÓN DEL SUPUESTO (TABLA ANOVA) ---
    # Usamos ANOVA Tipo II para evaluar la significancia de las interacciones
    # (C(type):age), (C(type):scholarship), y (C(type):C(gender)).
    ancova_interaction_results = sm.stats.anova_lm(ancova_interaction_model, typ=2)

    print("--- ANCOVA con interacciones (Tabla de verificación de supuestos) ---")
    print("El supuesto se cumple si el p-valor (PR(>F)) de las interacciones es > 0.05")
    print(ancova_interaction_results)
    return

In [4]:
# Función para realizar pruebas post-hoc con corrección de Bonferroni en medias ajustadas  # noqa: E501

def posthoc_contrasts_bonferroni(data, dv, between, ancova_model):

    # =======================================================================
    # 2. DEFINICIÓN Y APLICACIÓN DE CONTRASTES POST-HOC
    # =======================================================================

    # Identificar el orden de los coeficientes del modelo para construir la matriz L.
    # El orden es: [Intercept, C(type)[T.nonvictim], C(type)[T.victim], age, scholarship]  # noqa: E501
    coef_names = ancova_model.params.index.tolist()

    # Crear la matriz de contrastes (L) para las 3 comparaciones por pares
    # La matriz L tendrá 3 filas (comparaciones) y 5 columnas (coeficientes)
    L = np.zeros((3, len(coef_names)))

    cat_vars = data[between].unique().tolist()
    cat_vars = cat_vars[1:]  # Eliminar el grupo de referencia

    # Índice de los coeficientes de interés
    idx_level_1 = coef_names.index(f'C({between})[T.{cat_vars[0]}]')
    idx_level_2 = coef_names.index(f'C({between})[T.{cat_vars[1]}]')
    L[0, idx_level_1] = 1 
    L[1, idx_level_2] = 1
    L[2, idx_level_1] = 1
    L[2, idx_level_2] = -1

    # =======================================================================
    # 3. PRUEBA T Y CORRECCIÓN POR COMPARACIONES MÚLTIPLES
    # =======================================================================

    # Realizar las pruebas t con la matriz de contrastes
    t_tests = ancova_model.t_test(L)

    # Extraer los p-valores sin ajustar
    raw_p_values = t_tests.pvalue

    # Aplicar la corrección de Bonferroni
    reject, p_adjusted, _, _ = multipletests(
        raw_p_values,
        alpha=0.05,
        method='bonferroni'
    )

    # =======================================================================
    # 4. CREACIÓN Y VISUALIZACIÓN DE RESULTADOS
    # =======================================================================

    cat_vars = data[between].unique().tolist()

    comparisons = [
        f'{cat_vars[1]} vs. {cat_vars[0]}',
        f'{cat_vars[2]} vs. {cat_vars[0]}',
        f'{cat_vars[2]} vs. {cat_vars[1]}'
    ]

    results_df = pd.DataFrame({
        'Comparación': comparisons,
        'Diferencia_Media_Ajustada': t_tests.effect.round(4),
        'Estadístico_t': t_tests.tvalue.round(4),
        'p_valor_sin_ajustar': raw_p_values.round(4),
        'p_valor_Bonferroni': p_adjusted.round(4),
        'Significativo (alpha=0.05)': reject
    })

    print("--- Pruebas Post-Hoc (Bonferroni) en Medias Ajustadas ---")
    print(results_df)
    return

## Análisis covariables

In [5]:
data_subjects_erps = pd.read_csv('data_subjects.csv')
data_subjects_erps = data_subjects_erps.rename(columns={
    'Unnamed: 0': 'subject'
})

data_subjects_erps.head()

,subject,type,group,victim_self,exposure_level,age,gender,scholarship,laterality,IAT_score,mean_amplitude_co,mean_amplitude_in,mean_amplitude_dif,peak_amplitude_co,peak_amplitude_in,peak_latency_co,peak_latency_in
0,21100,excombatant,exguerrilla,yes,high,19,F,11,D,0.319044,0.586812,0.537628,0.049184,-0.117407,0.321850,0.160156,0.250000
1,21101,excombatant,exparamilitar,yes,high,46,M,11,D,-0.070836,0.463325,1.170137,-0.706813,0.400212,0.501799,0.210938,0.148438
2,21102,excombatant,exparamilitar,yes,low,31,M,11,D,0.488057,-4.757545,-3.940181,-0.817364,-5.105995,-4.638216,0.199219,0.207031
3,21105,excombatant,exparamilitar,yes,high,48,M,8,D,0.258352,0.410431,0.404159,0.006272,-0.007485,-0.122637,0.148438,0.167969
4,21107,excombatant,exguerrilla,no,high,25,M,11,D,-0.119566,0.256688,0.392556,-0.135867,0.018313,0.167643,0.230469,0.148438


In [6]:
data_subjects_erps = data_subjects_erps[
    data_subjects_erps['type'] != 'excombatant'
]

data_subjects_erps.info()

<class 'pandas.core.frame.DataFrame'>
Index: 37 entries, 29 to 86
Data columns (total 17 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   subject             37 non-null     int64  
 1   type                37 non-null     object 
 2   group               37 non-null     object 
 3   victim_self         35 non-null     object 
 4   exposure_level      37 non-null     object 
 5   age                 37 non-null     int64  
 6   gender              37 non-null     object 
 7   scholarship         37 non-null     int64  
 8   laterality          37 non-null     object 
 9   IAT_score           36 non-null     float64
 10  mean_amplitude_co   37 non-null     float64
 11  mean_amplitude_in   37 non-null     float64
 12  mean_amplitude_dif  37 non-null     float64
 13  peak_amplitude_co   37 non-null     float64
 14  peak_amplitude_in   37 non-null     float64
 15  peak_latency_co     37 non-null     float64
 16  peak_latency_i

### Age

In [7]:
# Normalidad de age por type
pg.normality(
    data=data_subjects_erps, 
    dv='age', 
    group='type', 
    method='shapiro', 
    alpha=0.05
    )

,W,pval,normal
type,,,
nonvictim,0.958400,0.664618,True
victim,0.960358,0.496554,True


In [8]:
# Homocedasticity de age por type
pg.homoscedasticity(
    data=data_subjects_erps, 
    dv='age', 
    group='type', 
    method='levene', 
    alpha=0.05
    )

,W,pval,equal_var
levene,5.469744,0.025186,False


In [9]:
# ANOVA de age por type
pg.anova(
    data=data_subjects_erps, 
    dv='age', 
    between='type', 
    effsize='np2'
    )

,Source,ddof1,ddof2,F,p_unc,np2
0,type,1,35,4.973195,0.032254,0.124413


In [10]:
pg.welch_anova(data=data_subjects_erps, dv='age', between='type')

,Source,ddof1,ddof2,F,p_unc,np2
0,type,1,33.879831,6.112067,0.018607,0.124413


**Conclusión**: Hay diferencias significativas en la edad por tipo de actor.

### Scholarship

In [11]:
# Normalidad de scholarship por type
pg.normality(
    data=data_subjects_erps, 
    dv='scholarship', 
    group='type', 
    method='shapiro', 
    alpha=0.05
    )

,W,pval,normal
type,,,
nonvictim,0.766030,0.001392,False
victim,0.868774,0.007428,False


In [12]:
# Homocedasticity de scholarship por type
pg.homoscedasticity(
    data=data_subjects_erps, 
    dv='scholarship', 
    group='type', 
    method='levene', 
    alpha=0.05
    )

,W,pval,equal_var
levene,1.848278,0.182679,True


In [13]:
# Kruskal para scholarship por type
pg.kruskal(
    data=data_subjects_erps, 
    dv='scholarship', 
    between='type'
    )

,Source,ddof1,H,p_unc
Kruskal,type,1,1.495294,0.221397


**Conclusión**: No hay diferencias significativas en los años de estudio por tipo de actor.

### Sex

In [14]:
# prueba chi-squared type vs gender
contingency_table = pd.crosstab(
    data_subjects_erps['type'], 
    data_subjects_erps['gender']
    )

chi2, p, dof, expected = chi2_contingency(contingency_table)
print(f"Chi2: {chi2}, p-value: {p}, dof: {dof}")

Chi2: 13.455354020979021, p-value: 0.0002443079909029722, dof: 1


**Conclusión**: Sí hay diferencias significativas en el sexo por tipo de actor.

## Análisis IAT

In [21]:
dv = 'IAT_score'
between = 'type'

ancova_model = test_ancova_full(
    data=data_subjects_erps, 
    dv=dv, 
    between=between
)

--- 1. Pruebas de Supuestos del ANCOVA ---


KeyError: 'residuals'

In [19]:
verify_ancova_assumptions(
    data=data_subjects_erps,
    dv=dv,
    between=between
)

--- ANCOVA con interacciones (Tabla de verificación de supuestos) ---
El supuesto se cumple si el p-valor (PR(>F)) de las interacciones es > 0.05
                       sum_sq    df         F    PR(>F)
C(type)              0.243253   1.0  2.881774  0.100677
C(gender)            0.009673   1.0  0.114600  0.737492
C(type):C(gender)    0.014957   1.0  0.177197  0.677007
age                  0.163499   1.0  1.936941  0.174959
C(type):age          0.000564   1.0  0.006685  0.935419
scholarship          0.015786   1.0  0.187009  0.668730
C(type):scholarship  0.121893   1.0  1.444044  0.239552
Residual             2.363507  28.0       NaN       NaN


**Conclusiones:**: No hay efecto de type cuando se ajusta por covariables.

## Análisis componente ERP

In [69]:
dv = 'mean_dif'
between = 'type'

ancova_model = test_ancova_full(
    data=data_subjects_erps, 
    dv=dv, 
    between=between
)

--- Tabla ANCOVA del Modelo Final ---
                 sum_sq    df         F    PR(>F)
C(type)       14.297530   2.0  4.983867  0.009093
C(gender)      5.058306   1.0  3.526473  0.063995
age            1.997301   1.0  1.392448  0.241446
scholarship    0.004662   1.0  0.003250  0.954678
Residual     116.184865  81.0       NaN       NaN
------------------------------------
--- Medias Ajustadas, Error Estándar e Intervalos de Confianza (95%) ---
          type  Media_Ajustada (M_adj)  Error_Estandar (EE)  IC_95%_Inferior  \
0  excombatant                  0.1193               0.1851          -0.2489   
1    nonvictim                  0.8197               0.3283           0.1664   
2       victim                 -0.7037               0.3213          -1.3430   

   IC_95%_Superior  
0           0.4876  
1           1.4730  
2          -0.0644  
------------------------------------
--- Tamaños del Efecto: Partial Eta-Squared (η²p) ---
        Efecto  Eta_Squared_Parcial (η²p)
0      C(type)

In [70]:
verify_ancova_assumptions(
    data=data_subjects_erps,
    dv=dv,
    between=between
)

--- ANCOVA con interacciones (Tabla de verificación de supuestos) ---
El supuesto se cumple si el p-valor (PR(>F)) de las interacciones es > 0.05
                         sum_sq    df         F    PR(>F)
C(type)               14.297530   2.0  4.919726  0.009826
C(gender)              2.907721   1.0  2.001071  0.161326
C(type):C(gender)      1.434993   2.0  0.493776  0.612288
age                    1.878758   1.0  1.292947  0.259125
C(type):age            5.034477   2.0  1.732344  0.183871
scholarship            0.015699   1.0  0.010804  0.917494
C(type):scholarship    1.460837   2.0  0.502668  0.606938
Residual             108.981150  75.0       NaN       NaN


**Conclusión:** si hay diferencias por tipo de actor.

In [71]:
posthoc_contrasts_bonferroni(
    data=data_subjects_erps,
    dv=dv,
    between=between,
    ancova_model=ancova_model
)

--- Pruebas Post-Hoc (Bonferroni) en Medias Ajustadas ---
                 Comparación  Diferencia_Media_Ajustada  Estadístico_t  \
0  nonvictim vs. excombatant                     0.7004         1.8737   
1     victim vs. excombatant                    -0.8230        -2.0299   
2       victim vs. nonvictim                     1.5235         3.1565   

   p_valor_sin_ajustar  p_valor_Bonferroni  Significativo (alpha=0.05)  
0               0.0646              0.1937                       False  
1               0.0456              0.1369                       False  
2               0.0022              0.0067                        True  


**Conclusiones**: Hay diferencias significativas entre no-víctimas y víctimas, ya ajustando por las covariables. En no-víctimas las amplitudes promedio de los ensayos del bloque congruente tienden a ser más grandes (positivas) que las del incongruente. En víctimas sucede lo contrario.

## ANOVA Mixto

In [72]:
def anova_mixto(data, within, between):

    data_long = pd.melt(
        data, 
        id_vars=['subject', between], 
        value_vars=within, 
        var_name='condition', 
        value_name='amplitude'
        )


    anova = pg.mixed_anova(
        dv='amplitude', 
        within='condition', 
        between=between, 
        subject='subject', 
        data=data_long,
        correction='auto'
        )
    
    posthoc = pg.pairwise_tests(
        dv='amplitude', 
        between=between, 
        within='condition', 
        data=data_long, 
        parametric=True, 
        subject='subject', 
        padjust='holm',
        effsize='cohen'
        )

    return anova, posthoc

In [73]:
# ANOVA MIXTO

within = ['mean_co', 'mean_in']
between = 'type'

omnibus, posthoc = anova_mixto(data=data_subjects_erps, within=within, between=between)
print(omnibus)

        Source        SS  DF1  DF2        MS         F     p_unc       np2  \
0         type  0.273336    2   84  0.136668  0.080698  0.922544  0.001918   
1    condition  0.044415    1   84  0.044415  0.059238  0.808297  0.000705   
2  Interaction  6.816419    2   84  3.408209  4.545664  0.013352  0.097660   

   eps  
0  NaN  
1  1.0  
2  NaN  


In [74]:
print(posthoc)

           Contrast condition            A          B Paired Parametric  \
0         condition         -      mean_co    mean_in   True       True   
1              type         -  excombatant  nonvictim  False       True   
2              type         -  excombatant     victim  False       True   
3              type         -    nonvictim     victim  False       True   
4  condition * type   mean_co  excombatant  nonvictim  False       True   
5  condition * type   mean_co  excombatant     victim  False       True   
6  condition * type   mean_co    nonvictim     victim  False       True   
7  condition * type   mean_in  excombatant  nonvictim  False       True   
8  condition * type   mean_in  excombatant     victim  False       True   
9  condition * type   mean_in    nonvictim     victim  False       True   

          T        dof alternative     p_unc    p_corr p_adjust   BF10  \
0  0.233936  86.000000   two-sided  0.815591       NaN      NaN  0.122   
1 -0.296686  25.989773   t